# TFT electricity runner for Colab

This notebook clones the repo, builds the dataset, runs a smoke test, then trains, predicts, evaluates, and compares models while saving outputs to Google Drive and locally in the repo.

In [ ]:
# Imports

import shutil
from pathlib import Path
import torch


In [ ]:
# Configuration

REPO_URL = "https://github.com/HannaVallner/tft_electricity.git"
REPO_DIR = "tft_electricity"

USE_DRIVE = True
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/tft_electricity_outputs"

RUN_SMOKE_TEST = True
RUN_FULL_TRAINING = False

# Random seeds to run. For quick smoke tests, this can be shortened, e.g. [0].
RANDOM_SEEDS = [0, 1, 2, 3, 4]

SMOKE_MODELS = ["baseline"]
FULL_MODELS = ["baseline", "no_lstm", "no_attention", "mlp_features", "transformer_only"]

SMOKE_TRAIN_IDS = 20
SMOKE_VALID_IDS = 20
SMOKE_EPOCHS = 2

FULL_EPOCHS = 100
PRINT_EVERY = 20


In [ ]:
# Clone repo and install requirements

!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!pip install -r requirements.txt

In [ ]:
# Mount Google Drive if enabled (optional)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# Create output folders in Google Drive if enabled

LOCAL_OUTPUT_ROOT = Path("outputs")
LOCAL_CHECKPOINTS = LOCAL_OUTPUT_ROOT / "checkpoints"
LOCAL_PREDICTIONS = LOCAL_OUTPUT_ROOT / "predictions"
LOCAL_METRICS = LOCAL_OUTPUT_ROOT / "metrics"
LOCAL_PLOTS = LOCAL_OUTPUT_ROOT / "plots"

for p in [LOCAL_CHECKPOINTS, LOCAL_PREDICTIONS, LOCAL_METRICS, LOCAL_PLOTS]:
    p.mkdir(parents=True, exist_ok=True)

if USE_DRIVE:
    DRIVE_OUTPUT_ROOT = Path(DRIVE_OUTPUT_ROOT)
    DRIVE_CHECKPOINTS = DRIVE_OUTPUT_ROOT / "checkpoints"
    DRIVE_PREDICTIONS = DRIVE_OUTPUT_ROOT / "predictions"
    DRIVE_METRICS = DRIVE_OUTPUT_ROOT / "metrics"
    DRIVE_PLOTS = DRIVE_OUTPUT_ROOT / "plots"
    for p in [DRIVE_CHECKPOINTS, DRIVE_PREDICTIONS, DRIVE_METRICS, DRIVE_PLOTS]:
        p.mkdir(parents=True, exist_ok=True)

def copy_to_drive(local_path: Path, drive_path: Path):
    if USE_DRIVE and local_path.exists():
        shutil.copy2(local_path, drive_path)


In [ ]:
# Check GPU

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Build dataset if not yet present

data_path = Path("data/electricity_processed.csv")
if not data_path.exists():
    !python src/create_dataset.py
else:
    print(f"Dataset already exists: {data_path}")

In [ ]:
# Run a smoke test if enabled

if RUN_SMOKE_TEST:
    for seed in RANDOM_SEEDS:
        for model_name in SMOKE_MODELS:
            print(f"===== SMOKE TEST: {model_name} | seed {seed} =====")

            ckpt = Path(f"outputs/checkpoints/{model_name}_seed_{seed}_best.pt")
            hist = Path(f"outputs/metrics/{model_name}_seed_{seed}_training_history.json")

            if ckpt.exists() and hist.exists():
                print(f"Skipping {model_name} seed {seed}: checkpoint and history already exist.")
            else:
                !PYTHONPATH=/content/tft_electricity:/content/tft_electricity/src python src/train.py \
                    --model {model_name} \
                    --data_path data/electricity_processed.csv \
                    --save_dir outputs/checkpoints \
                    --metrics_dir outputs/metrics \
                    --num_epochs {SMOKE_EPOCHS} \
                    --num_train_ids {SMOKE_TRAIN_IDS} \
                    --num_valid_ids {SMOKE_VALID_IDS} \
                    --print_every {PRINT_EVERY} \
                    --seed {seed}

            if USE_DRIVE:
                copy_to_drive(ckpt, DRIVE_CHECKPOINTS / ckpt.name)
                copy_to_drive(hist, DRIVE_METRICS / hist.name)


In [ ]:
# Full training if enabled

if RUN_FULL_TRAINING:
    for seed in RANDOM_SEEDS:
        for model_name in FULL_MODELS:
            print(f"===== FULL TRAINING: {model_name} | seed {seed} =====")

            ckpt = Path(f"outputs/checkpoints/{model_name}_seed_{seed}_best.pt")
            hist = Path(f"outputs/metrics/{model_name}_seed_{seed}_training_history.json")

            if ckpt.exists() and hist.exists():
                print(f"Skipping {model_name} seed {seed}: checkpoint and history already exist.")
            else:
                !PYTHONPATH=/content/tft_electricity:/content/tft_electricity/src python src/train.py \
                    --model {model_name} \
                    --data_path data/electricity_processed.csv \
                    --save_dir outputs/checkpoints \
                    --metrics_dir outputs/metrics \
                    --num_epochs {FULL_EPOCHS} \
                    --print_every {PRINT_EVERY} \
                    --seed {seed}

            if USE_DRIVE:
                copy_to_drive(ckpt, DRIVE_CHECKPOINTS / ckpt.name)
                copy_to_drive(hist, DRIVE_METRICS / hist.name)


In [ ]:
# Predict for available checkpoints

models_to_predict = FULL_MODELS if RUN_FULL_TRAINING else SMOKE_MODELS

for seed in RANDOM_SEEDS:
    for model_name in models_to_predict:
        ckpt = Path(f"outputs/checkpoints/{model_name}_seed_{seed}_best.pt")
        if not ckpt.exists():
            print(f"Skipping {model_name} seed {seed}: checkpoint not found.")
            continue

        output_path = Path(f"outputs/predictions/{model_name}_seed_{seed}_predictions.csv")

        if output_path.exists():
            print(f"Skipping prediction for {model_name} seed {seed}: predictions already exist.")
        else:
            print(f"===== PREDICT: {model_name} | seed {seed} =====")
            !PYTHONPATH=/content/tft_electricity:/content/tft_electricity/src python src/predict.py \
                --model {model_name} \
                --data_path data/electricity_processed.csv \
                --checkpoint_path {ckpt} \
                --output_path {output_path} \
                --batch_size 64

        if USE_DRIVE:
            copy_to_drive(output_path, DRIVE_PREDICTIONS / output_path.name)


In [ ]:
# Evaluate each model and seed

for seed in RANDOM_SEEDS:
    for model_name in models_to_predict:
        ckpt = Path(f"outputs/checkpoints/{model_name}_seed_{seed}_best.pt")
        pred = Path(f"outputs/predictions/{model_name}_seed_{seed}_predictions.csv")
        metrics_path = Path(f"outputs/metrics/{model_name}_seed_{seed}_metrics.json")

        if not ckpt.exists() or not pred.exists():
            print(f"Skipping {model_name} seed {seed}: missing checkpoint or predictions.")
            continue

        if metrics_path.exists():
            print(f"Skipping evaluation for {model_name} seed {seed}: metrics already exist.")
        else:
            print(f"===== EVALUATE: {model_name} | seed {seed} =====")
            !PYTHONPATH=/content/tft_electricity:/content/tft_electricity/src python src/evaluate.py \
                --model {model_name} \
                --data_path data/electricity_processed.csv \
                --checkpoint_path {ckpt} \
                --predictions_path {pred} \
                --metrics_path {metrics_path} \
                --plots_dir outputs/plots \
                --batch_size 64 \
                --seed {seed} 

        if USE_DRIVE:
            copy_to_drive(metrics_path, DRIVE_METRICS / metrics_path.name)
            for plot_path in Path("outputs/plots").glob(f"{model_name}_seed_{seed}_*.png"):
                copy_to_drive(plot_path, DRIVE_PLOTS / plot_path.name)
            comparison_plot = Path(f"outputs/model_comparison/all_models_seed_{seed}_p50_comparison.png")
            if comparison_plot.exists():
                copy_to_drive(comparison_plot, DRIVE_PLOTS / comparison_plot.name)


In [ ]:
# Compare models across seeds

!PYTHONPATH=/content/tft_electricity:/content/tft_electricity/src python src/compare_models.py     --metrics_dir outputs/metrics     --plots_dir outputs/plots

if USE_DRIVE:
    for comparison_file in Path("outputs/metrics").glob("model_comparison*.csv"):
        copy_to_drive(comparison_file, DRIVE_METRICS / comparison_file.name)

    for comparison_file in Path("outputs/metrics").glob("model_comparison*.tex"):
        copy_to_drive(comparison_file, DRIVE_METRICS / comparison_file.name)

    for plot_file in Path("outputs/plots").glob("compare_*.png"):
        copy_to_drive(plot_file, DRIVE_PLOTS / plot_file.name)


In [ ]:
# Zip outputs for download

!zip -r tft_outputs.zip outputs
print("Created tft_outputs.zip")